In [4]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Extract MP3 Cover Art → Image) -----######-----###### #
import os
from mutagen.id3 import ID3, APIC

def _coverart_0109_extract_GET_img_from_mp3(mp3_path):
    """
    Extracts cover art (if available) from an MP3 file and saves it as a PNG in the same folder.

    Parameters:
        mp3_path: str, full path to the .mp3 file
    Returns:
        str path of the extracted image, or None if no cover art found
    """
    try:
        if not os.path.isfile(mp3_path):
            raise FileNotFoundError(f"File not found: {mp3_path}")

        # load ID3 tags
        tags = ID3(mp3_path)

        # find first APIC frame (cover art)
        for frame in tags.getall("APIC"):
            img_data = frame.data
            # choose extension based on MIME
            if frame.mime.lower() in ["image/jpeg", "image/jpg"]:
                ext = ".jpg"
            else:
                ext = ".png"

            out_path = os.path.splitext(mp3_path)[0] + "_cover" + ext
            with open(out_path, "wb") as f:
                f.write(img_data)

            return out_path

        return None  # no APIC frames

    except Exception as e:
        print(f"⚠️ Error extracting cover art: {e}")
        return None


In [5]:
mp3_path = "/Users/yerik/Desktop/a.mp3"

out_img = _coverart_0109_extract_GET_img_from_mp3(mp3_path)
print("✅ Saved cover art at:", out_img)


✅ Saved cover art at: /Users/yerik/Desktop/a_cover.jpg


In [8]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Batch: Overlay at Mid-Top 1/9 → Re-Embed) -----######-----###### #
import os, io, math
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from mutagen.id3 import ID3, APIC, ID3NoHeaderError

def _coverart_0109_overlayembed_GET_mp3_updated(
    mp3_path,
    overlay_png_path,
    place="mid_top",
    area_fraction=1/9,
    out_format="PNG",
    quality=92,
    keep_if_no_cover=True,
    y_center_ratio=1/6  # mid-top anchor; smaller = higher
):
    try:
        mp3_path = str(mp3_path)
        overlay_png_path = str(overlay_png_path)
        if not os.path.isfile(mp3_path):
            return {"status":"fail","error":f"File not found: {mp3_path}","mp3_path":mp3_path}
        if not os.path.isfile(overlay_png_path):
            return {"status":"fail","error":f"Overlay PNG not found: {overlay_png_path}","mp3_path":mp3_path}

        try:
            tags = ID3(mp3_path)
        except ID3NoHeaderError:
            tags = ID3()
            tags.save(mp3_path)
            tags = ID3(mp3_path)

        apic_frames = tags.getall("APIC")
        if apic_frames:
            base_data = apic_frames[0].data
            base_img = Image.open(io.BytesIO(base_data)).convert("RGBA")
        else:
            if not keep_if_no_cover:
                return {"status":"fail","error":"No embedded cover found and keep_if_no_cover=False","mp3_path":mp3_path}
            ov_hint = Image.open(overlay_png_path).convert("RGBA")
            side = max(ov_hint.size[0], ov_hint.size[1], 1000)
            base_img = Image.new("RGBA", (side, side), (0,0,0,255))

        bw, bh = base_img.size

        overlay = Image.open(overlay_png_path).convert("RGBA")
        ow0, oh0 = overlay.size
        if ow0 == 0 or oh0 == 0:
            return {"status":"fail","error":"Overlay image has zero dimension","mp3_path":mp3_path}

        target_area = (bw * bh) * float(area_fraction)
        s = math.sqrt(max(target_area, 1.0) / (ow0 * oh0))
        new_w = max(1, min(bw, int(round(ow0 * s))))
        new_h = max(1, min(bh, int(round(oh0 * s))))
        aspect = ow0 / oh0
        if abs((new_w / new_h) - aspect) > 1e-3:
            new_h = max(1, int(round(new_w / aspect)))
            if new_h > bh:
                new_h = bh
                new_w = max(1, int(round(new_h * aspect)))

        overlay_resized = overlay.resize((new_w, new_h), Image.LANCZOS)

        x = (bw - new_w) // 2
        y_center = bh * float(y_center_ratio)   # <-- adjust this to nudge higher/lower
        y = int(round(y_center - new_h / 2))
        y = max(0, min(bh - new_h, y))

        composited = base_img.copy()
        composited.alpha_composite(overlay_resized, (x, y))

        buf = io.BytesIO()
        if out_format.upper() == "JPEG":
            composited.convert("RGB").save(buf, format="JPEG", quality=int(quality), optimize=True)
            mime = "image/jpeg"
        else:
            composited.save(buf, format="PNG", optimize=True)
            mime = "image/png"
        img_bytes = buf.getvalue()
        buf.close()

        if tags.getall("APIC"):
            tags.delall("APIC")
        tags.add(APIC(encoding=3, mime=mime, type=3, desc="Cover", data=img_bytes))
        tags.save(mp3_path)

        return {"status":"ok","error":None,"mp3_path":mp3_path,"final_bytes":len(img_bytes),"mime":mime}

    except Exception as e:
        return {"status":"fail","error":str(e),"mp3_path":mp3_path}


def _coverart_0109_overlayembed_GET_df_batch(
    paths_or_df,
    overlay_png_path,
    col_name="Path",
    area_fraction=1/9,
    y_center_ratio=1/6,
    out_format="PNG",
    quality=92,
    keep_if_no_cover=True
):
    """
    Batch-apply the mid-top 1/9 overlay+embed to many MP3s.
    Input:
      - paths_or_df: DataFrame with column `col_name` or an iterable of paths
      - overlay_png_path: single PNG to overlay on all items
    Output:
      - list of dict logs (status, error, mp3_path, final_bytes, mime)
      - If a DataFrame is passed, also adds columns: 'status', 'error', 'mime', 'final_bytes'
        and returns the same DataFrame.
    """
    # Normalize inputs → list of paths
    try:
        import pandas as pd
        is_df = hasattr(paths_or_df, "iloc")
    except Exception:
        pd = None
        is_df = False

    if is_df:
        paths = list(paths_or_df[col_name].astype(str).values)
    else:
        paths = [str(p) for p in paths_or_df]

    logs = []
    pbar = tqdm(total=len(paths), desc="TQM: overlay+embed (mid-top 1/9)", unit="file")
    for p in paths:
        res = _coverart_0109_overlayembed_GET_mp3_updated(
            mp3_path=p,
            overlay_png_path=overlay_png_path,
            place="mid_top",
            area_fraction=area_fraction,
            out_format=out_format,
            quality=quality,
            keep_if_no_cover=keep_if_no_cover,
            y_center_ratio=y_center_ratio
        )
        logs.append(res)
        pbar.update(1)
    pbar.close()

    if is_df:
        # Attach results back to df in order
        status = [r.get("status") for r in logs]
        error  = [r.get("error") for r in logs]
        mime   = [r.get("mime") for r in logs]
        fbytes = [r.get("final_bytes") for r in logs]
        paths_or_df["status"] = status
        paths_or_df["error"] = error
        paths_or_df["mime"] = mime
        paths_or_df["final_bytes"] = fbytes
        return paths_or_df

    return logs


In [12]:
res = _coverart_0109_overlayembed_GET_mp3_updated(
    mp3_path="/Users/yerik/Desktop/a.mp3",
    overlay_png_path="/Users/yerik/Desktop/sinc.png",
    out_format="PNG",
    quality=92,
    keep_if_no_cover=True
)


# ALL MP3 

In [1]:
#### ALL MP3 

png_path = "/Users/yerik/Desktop/sinc.png"

In [2]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Recursive Folder Scan → Overlay All MP3s) -----######-----###### #
import os, math, io
import pandas as pd
from tqdm import tqdm
from PIL import Image
from mutagen.id3 import ID3, APIC, ID3NoHeaderError

def _coverart_0209_overlayembed_walk_GET_df_status(
    root_folder,
    overlay_png_path,
    area_fraction=1/9,       # overlay covers 1/9 of base cover
    y_center_frac=1/6,       # vertical center of overlay (1/6 ≈ middle-up)
    out_format="PNG",        # or "JPEG"
    quality=92,
    keep_if_no_cover=True
):
    """
    Recursively walks `root_folder`, finds all .mp3, overlays `overlay_png_path` at mid-top,
    and re-embeds cover. Returns a DataFrame log.
    """

    # collect mp3 paths
    mp3_paths = []
    for dirpath, _, filenames in os.walk(root_folder):
        for fn in filenames:
            if fn.lower().endswith(".mp3") and not fn.startswith("._"):
                mp3_paths.append(os.path.join(dirpath, fn))

    if not mp3_paths:
        raise FileNotFoundError(f"No .mp3 files found in {root_folder}")

    # preload overlay
    overlay = Image.open(overlay_png_path).convert("RGBA")
    ow0, oh0 = overlay.size
    if ow0 == 0 or oh0 == 0:
        raise ValueError("Overlay image has zero dimension.")

    results = []
    pbar = tqdm(total=len(mp3_paths), desc="TQM | Recursive MP3 overlay+embed", unit="file")
    try:
        for mp3_path in mp3_paths:
            status, error, final_bytes, mime = "fail", None, None, None
            try:
                # ensure ID3 exists
                try:
                    tags = ID3(mp3_path)
                except ID3NoHeaderError:
                    tags = ID3(); tags.save(mp3_path); tags = ID3(mp3_path)

                # base cover
                apic_frames = tags.getall("APIC")
                if apic_frames:
                    base_data = apic_frames[0].data
                    base_img = Image.open(io.BytesIO(base_data)).convert("RGBA")
                else:
                    if not keep_if_no_cover:
                        raise RuntimeError("No embedded cover and keep_if_no_cover=False")
                    base_img = Image.new("RGBA", (1000, 1000), (0,0,0,255))

                bw, bh = base_img.size

                # scale overlay to target area
                target_area = (bw * bh) * float(area_fraction)
                s = math.sqrt(max(target_area, 1.0) / (ow0 * oh0))
                new_w = max(1, min(bw, int(round(ow0 * s))))
                new_h = max(1, min(bh, int(round(oh0 * s))))
                aspect = ow0 / oh0
                if abs((new_w / new_h) - aspect) > 1e-3:
                    new_h = max(1, int(round(new_w / aspect)))
                    if new_h > bh:
                        new_h = bh
                        new_w = max(1, int(round(new_h * aspect)))
                overlay_resized = overlay.resize((new_w, new_h), Image.LANCZOS)

                # position: centered X, y_center_frac of height
                x = (bw - new_w) // 2
                y_center = bh * float(y_center_frac)
                y = int(round(y_center - new_h / 2))
                y = max(0, min(bh - new_h, y))

                # composite
                comp = base_img.copy()
                comp.alpha_composite(overlay_resized, (x, y))

                # encode
                buf = io.BytesIO()
                if out_format.upper() == "JPEG":
                    comp.convert("RGB").save(buf, format="JPEG", quality=int(quality), optimize=True)
                    mime = "image/jpeg"
                else:
                    comp.save(buf, format="PNG", optimize=True)
                    mime = "image/png"
                img_bytes = buf.getvalue()
                buf.close()

                # replace APIC
                if tags.getall("APIC"):
                    tags.delall("APIC")
                tags.add(APIC(encoding=3, mime=mime, type=3, desc="Cover", data=img_bytes))
                tags.save(mp3_path)

                status = "ok"
                final_bytes = len(img_bytes)

            except Exception as e:
                error = str(e)

            results.append({
                "Path": mp3_path,
                "status": status,
                "error": error,
                "final_bytes": final_bytes,
                "mime": mime
            })
            pbar.update(1)
    finally:
        pbar.close()

    df_log = pd.DataFrame(results)
    return df_log


In [4]:
root_folder = "/Users/yerik/Music/_1_NEW_SOURCE/_25_ADD_SIN_good"                     # 🔥 set your top-level folder
overlay_png_path = png_path # 🔥 set overlay image

df_log = _coverart_0209_overlayembed_walk_GET_df_status(
    root_folder=root_folder,
    overlay_png_path=overlay_png_path,
    area_fraction=1/9,     # overlay covers 1/9 of cover
    y_center_frac=1/6,     # “middle-up” placement
    out_format="PNG",      # or "JPEG"
    quality=92,
    keep_if_no_cover=True
)

import caas_jupyter_tools
caas_jupyter_tools.display_dataframe_to_user("MP3 Overlay Log", df_log)


TQM | Recursive MP3 overlay+embed: 100%|████████████████████████████████| 1263/1263 [17:04<00:00,  1.23file/s]


ModuleNotFoundError: No module named 'caas_jupyter_tools'

## apparently better 

In [5]:
# -----######-----###### CORE IMPORTABLE FUNCTION (Recursive MP3 Cover Overlay • Mid-Top 1/9 • Front APIC Only) -----######-----###### #
import os, io, math
import pandas as pd
from tqdm import tqdm
from PIL import Image
from mutagen.id3 import ID3, APIC, ID3NoHeaderError

def _coverart_0109_overlayembed_walk_GET_df_status(
    root_folder,
    overlay_png_path,
    area_fraction=1/9,       # overlay covers 1/9 of base cover area
    y_center_frac=1/6,       # vertical center position (0=top, 1=bottom). 1/6 ≈ “middle-up”
    out_format="PNG",        # "PNG" or "JPEG" for embedded cover
    quality=92,              # JPEG quality if used
    keep_if_no_cover=True,   # if no cover exists, build a black base and place overlay
    export_csv_path=None     # if set to a filepath, writes the log CSV
):
    """
    Recursively overlays a PNG on every MP3 cover (mid-top, 1/9 area), replacing only the FRONT cover (APIC type=3).
    Preserves all other ID3 frames and non-front APICs.

    Returns:
        DataFrame with columns: Path, status, error, final_bytes, mime
    """
    # ---- collect mp3s ----
    mp3_paths = []
    for dirpath, _, filenames in os.walk(root_folder):
        for fn in filenames:
            if fn.lower().endswith(".mp3") and not fn.startswith("._"):
                mp3_paths.append(os.path.join(dirpath, fn))

    if not mp3_paths:
        return pd.DataFrame([{
            "Path": None, "status": "fail", "error": f"No .mp3 files found in {root_folder}", "final_bytes": None, "mime": None
        }])

    # ---- preload overlay once ----
    overlay = Image.open(overlay_png_path).convert("RGBA")
    ow0, oh0 = overlay.size
    if ow0 == 0 or oh0 == 0:
        raise ValueError("Overlay image has zero dimension.")

    results = []
    pbar = tqdm(total=len(mp3_paths), desc="TQM | Recursive MP3 overlay+embed", unit="file")
    try:
        for mp3_path in mp3_paths:
            status, error, final_bytes, mime = "fail", None, None, None
            try:
                # ensure ID3 exists
                try:
                    tags = ID3(mp3_path)
                except ID3NoHeaderError:
                    tags = ID3(); tags.save(mp3_path); tags = ID3(mp3_path)

                # ---- base cover (prefer FRONT APIC type=3 if present, else first APIC) ----
                apics = tags.getall("APIC")
                base_img = None
                base_data = None

                # find front cover first
                front_idx = None
                for i, fr in enumerate(apics):
                    # Mutagen APIC.type: 3 == front cover
                    if getattr(fr, "type", None) == 3:
                        front_idx = i
                        break

                if front_idx is not None:
                    base_data = apics[front_idx].data
                elif apics:
                    base_data = apics[0].data

                if base_data is not None:
                    base_img = Image.open(io.BytesIO(base_data)).convert("RGBA")
                else:
                    if not keep_if_no_cover:
                        raise RuntimeError("No embedded cover and keep_if_no_cover=False")
                    # build a clean square base (quality floor 1000 px)
                    base_img = Image.new("RGBA", (1000, 1000), (0, 0, 0, 255))

                bw, bh = base_img.size

                # ---- scale overlay to target area (1/9) while preserving aspect ----
                target_area = (bw * bh) * float(area_fraction)
                s = math.sqrt(max(target_area, 1.0) / (ow0 * oh0))
                new_w = max(1, min(bw, int(round(ow0 * s))))
                new_h = max(1, min(bh, int(round(oh0 * s))))
                # exact aspect keep
                aspect = ow0 / oh0
                if abs((new_w / new_h) - aspect) > 1e-3:
                    new_h = max(1, int(round(new_w / aspect)))
                    if new_h > bh:
                        new_h = bh
                        new_w = max(1, int(round(new_h * aspect)))

                overlay_resized = overlay.resize((new_w, new_h), Image.LANCZOS)

                # ---- position mid-top (center X, y_center_frac) ----
                x = (bw - new_w) // 2
                y_center = bh * float(y_center_frac)
                y = int(round(y_center - new_h / 2))
                y = max(0, min(bh - new_h, y))  # clamp into canvas

                # ---- composite ----
                comp = base_img.copy()
                comp.alpha_composite(overlay_resized, (x, y))

                # ---- encode for APIC ----
                buf = io.BytesIO()
                if out_format.upper() == "JPEG":
                    comp.convert("RGB").save(buf, format="JPEG", quality=int(quality), optimize=True)
                    mime = "image/jpeg"
                else:
                    comp.save(buf, format="PNG", optimize=True)
                    mime = "image/png"
                img_bytes = buf.getvalue()
                buf.close()

                # ---- replace ONLY the FRONT cover; preserve other APICs ----
                # remove old front(s), keep others
                kept_apics = []
                for fr in apics:
                    if getattr(fr, "type", None) == 3:
                        # skip (we'll replace)
                        continue
                    kept_apics.append(fr)
                # clear all APICs then re-add kept + new front
                if apics:
                    tags.delall("APIC")
                for fr in kept_apics:
                    tags.add(fr)

                tags.add(APIC(
                    encoding=3,       # UTF-8
                    mime=mime,
                    type=3,           # front cover
                    desc="Cover",
                    data=img_bytes
                ))
                tags.save(mp3_path)

                status = "ok"
                final_bytes = len(img_bytes)

            except Exception as e:
                error = str(e)

            results.append({
                "Path": mp3_path,
                "status": status,
                "error": error,
                "final_bytes": final_bytes,
                "mime": mime
            })
            pbar.update(1)
    finally:
        pbar.close()

    df_log = pd.DataFrame(results)

    # optional export
    if export_csv_path:
        try:
            df_log.to_csv(export_csv_path, index=False)
        except Exception as e:
            # don't hard-fail just because CSV write failed
            print(f"⚠️ Could not write CSV log: {e}")

    return df_log


In [ ]:
# Set your inputs
root_folder = "/Users/yerik/Desktop"                      # <- top-level folder to scan
overlay_png_path = "/Users/yerik/Desktop/your_overlay.png"  # <- your PNG overlay

# Optional knobs
area_fraction = 1/9
y_center_frac = 1/6
out_format = "PNG"    # or "JPEG"
quality = 92
keep_if_no_cover = True
export_csv_path = None  # e.g., os.path.join(root_folder, "overlay_log.csv")

# Run
df_log = _coverart_0109_overlayembed_walk_GET_df_status(
    root_folder=root_folder,
    overlay_png_path=overlay_png_path,
    area_fraction=area_fraction,
    y_center_frac=y_center_frac,
    out_format=out_format,
    quality=quality,
    keep_if_no_cover=keep_if_no_cover,
    export_csv_path=export_csv_path
)

# Summary (no external deps)
print("=== SUMMARY ===")
print(df_log["status"].value_counts(dropna=False))
print("\nFirst 5:\n", df_log.head())
print("\nLast 5:\n", df_log.tail())
print("\nErrors (if any):")
print(df_log[df_log["status"] != "ok"][["Path", "error"]].head(20))
